# Phase 3 Outcome-Blind Lock Repair

This notebook reviews the completed, separately versioned Phase 3 lock repair.
It does **not** fit models, open OASIS-3 participant-level data, merge external
outcomes, or resume Phase 4.

The independent verifier is implemented separately in
`scripts/phase3_lock_repair_qc.py`.

In [1]:
from pathlib import Path
import json
import platform

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / "models" / "phase3").exists():
    ROOT = Path.cwd().resolve().parents[1]

LOCK = ROOT / "models" / "phase3" / "phase4_lock_v2"
ORIGINAL_LOCK = ROOT / "models" / "phase3" / "phase4_lock"
TABLES = ROOT / "results" / "csvs" / "phase3_lock_repair"
REPORTS = ROOT / "reports" / "phase3_lock_repair"

print(f"Python: {platform.python_version()}")
print(f"Original immutable lock: {ORIGINAL_LOCK}")
print(f"Repaired lock: {LOCK}")
print(f"Repaired lock exists: {LOCK.exists()}")

Python: 3.10.20
Original immutable lock: /media/faizaan/4TB/1_DATA_PROJECTS/2026/July clinical data adni and oasis and aibl/models/phase3/phase4_lock
Repaired lock: /media/faizaan/4TB/1_DATA_PROJECTS/2026/July clinical data adni and oasis and aibl/models/phase3/phase4_lock_v2
Repaired lock exists: True


## Final run status

In [2]:
run_metadata = json.loads(
    (TABLES / "phase3_lock_repair_run_metadata.json").read_text()
)
display(pd.Series(run_metadata, name="value").to_frame())

,value
b1_d1_equivalence_pass,True
copied_models,"[B1, D1]"
d3_temporal_pass,True
development_cohort,ADNI
final_decision,RESUME PHASE 4 WITH RESTRICTIONS
fitted_models,"[D0, B3, D3]"
independent_qc_checks,18
independent_qc_completed_at_utc,2026-07-29T02:37:21.308871+00:00
independent_qc_failures,0
independent_qc_pass,True


## Complete frozen-model inventory

In [3]:
inventory = pd.read_csv(LOCK / "model_inventory.csv")
display(
    inventory[
        [
            "model_id", "family", "horizon_months", "model_origin",
            "training_rows", "training_participants",
            "improvement_event_rows", "progression_event_rows",
            "improvement_alpha", "progression_alpha",
            "refit_during_repair", "sha256", "bytes",
        ]
    ].sort_values(["model_id", "horizon_months"]).reset_index(drop=True)
)
print(f"Frozen bundles: {len(inventory)}")

,model_id,family,horizon_months,model_origin,training_rows,training_participants,improvement_event_rows,progression_event_rows,improvement_alpha,progression_alpha,refit_during_repair,sha256,bytes
0,B1,baseline,12,verified_copy_from_original_lock,2447,2447,65,89,10.0,10.0,False,b408adbf1644174870ad96a4df3df633e9a3f53c6b2a5a...,8692
1,B1,baseline,24,verified_copy_from_original_lock,2447,2447,173,218,10.0,10.0,False,5cd4e5e45316f20bee3a3c0d88b11d9bf11b745f20bc46...,11316
2,B1,baseline,36,verified_copy_from_original_lock,2447,2447,219,302,10.0,10.0,False,52baf11e3cd593770ce25145a08fff317264aa0f95e972...,12965
3,B1,baseline,60,verified_copy_from_original_lock,2447,2447,280,410,10.0,100.0,False,e3d9f777e6cdaacc45e7ae91f1273de08f97e6879cd35f...,15286
4,B3,baseline,12,adni_only_outcome_blind_lock_repair,2447,2447,65,89,10.0,10.0,True,36fa8fe7d5047909bcef6ca3ded4b7f5621ce8de395c97...,9165
5,B3,baseline,36,adni_only_outcome_blind_lock_repair,2447,2447,219,302,10.0,10.0,True,6a5fdaad7f1554bc588e22e7c8042fa4ef099e740307f4...,13438
6,D0,dynamic,12,adni_only_outcome_blind_lock_repair,4517,1412,154,303,10.0,100.0,True,df4ce2b2ef7a32369c81b582eb81d8426aa6d807861013...,11790
7,D0,dynamic,36,adni_only_outcome_blind_lock_repair,4517,1412,503,1020,100.0,100.0,True,5d13f2f90552eadb03d947a13894ae6c583cbdf0cb6034...,20927
8,D1,dynamic,12,verified_copy_from_original_lock,4517,1412,154,303,100.0,100.0,False,f2e05fbfe91fdfe31ee46dcd674c09ffa46be7262ab299...,11493
9,D1,dynamic,24,verified_copy_from_original_lock,4517,1412,369,734,100.0,100.0,False,14cd2e62ca7ee94e0cdb749d1ecab981c2d14e643428b2...,16502


Frozen bundles: 14


In [4]:
required = pd.MultiIndex.from_tuples(
    [
        ("B1", 12), ("B1", 24), ("B1", 36), ("B1", 60),
        ("D0", 12), ("D0", 36),
        ("D1", 12), ("D1", 24), ("D1", 36), ("D1", 60),
        ("B3", 12), ("B3", 36),
        ("D3", 12), ("D3", 36),
    ],
    names=["model_id", "horizon_months"],
)
present = pd.MultiIndex.from_frame(
    inventory[["model_id", "horizon_months"]]
)
coverage = (
    pd.DataFrame(index=required)
    .assign(present=[item in present for item in required])
    .reset_index()
)
display(coverage)
print(f"Every required model present: {coverage['present'].all()}")

,model_id,horizon_months,present
0,B1,12,True
1,B1,24,True
2,B1,36,True
3,B1,60,True
4,D0,12,True
5,D0,36,True
6,D1,12,True
7,D1,24,True
8,D1,36,True
9,D1,60,True


Every required model present: True


## Recovered Phase 3 penalties — no retuning

In [5]:
penalties = pd.read_csv(LOCK / "penalty_source_recovery.csv")
display(
    penalties[
        [
            "model_id", "horizon_months", "cause",
            "recovered_final_alpha", "outer_fold_selected_alphas",
            "recovery_rule", "source_path", "retuned_during_repair",
        ]
    ]
)

,model_id,horizon_months,cause,recovered_final_alpha,outer_fold_selected_alphas,recovery_rule,source_path,retuned_during_repair
0,D0,12,improvement,10.0,0:100|1:10|2:100|3:10|4:10,modal outer-fold selected alpha; ties choose t...,results/csvs/phase3/fold_specific_ridge_penalt...,False
1,D0,12,progression,100.0,0:10|1:100|2:100|3:100|4:100,modal outer-fold selected alpha; ties choose t...,results/csvs/phase3/fold_specific_ridge_penalt...,False
2,D0,36,improvement,100.0,0:100|1:100|2:100|3:100|4:100,modal outer-fold selected alpha; ties choose t...,results/csvs/phase3/fold_specific_ridge_penalt...,False
3,D0,36,progression,100.0,0:100|1:100|2:100|3:100|4:100,modal outer-fold selected alpha; ties choose t...,results/csvs/phase3/fold_specific_ridge_penalt...,False
4,B3,12,improvement,10.0,0:100|1:10|2:10|3:10|4:10,modal outer-fold selected alpha; ties choose t...,results/csvs/phase3/fold_specific_ridge_penalt...,False
5,B3,12,progression,10.0,0:10|1:10|2:10|3:10|4:10,modal outer-fold selected alpha; ties choose t...,results/csvs/phase3/fold_specific_ridge_penalt...,False
6,B3,36,improvement,10.0,0:10|1:10|2:10|3:10|4:10,modal outer-fold selected alpha; ties choose t...,results/csvs/phase3/fold_specific_ridge_penalt...,False
7,B3,36,progression,10.0,0:10|1:100|2:10|3:10|4:100,modal outer-fold selected alpha; ties choose t...,results/csvs/phase3/fold_specific_ridge_penalt...,False
8,D3,12,improvement,10.0,0:10|1:10|2:10|3:10|4:10,modal outer-fold selected alpha; ties choose t...,results/csvs/phase3/fold_specific_ridge_penalt...,False
9,D3,12,progression,100.0,0:10|1:100|2:100|3:100|4:100,modal outer-fold selected alpha; ties choose t...,results/csvs/phase3/fold_specific_ridge_penalt...,False


## D0, B3, and D3 ADNI training metadata

In [6]:
training = pd.read_csv(LOCK / "repaired_model_training_metadata.csv")
display(
    training[
        [
            "model_id", "family", "horizon_months", "dataset_variant",
            "training_rows", "training_participants",
            "improvement_event_rows", "progression_event_rows",
            "improvement_alpha", "progression_alpha",
            "continuous_features", "categorical_features",
            "endpoint_construction", "baseline_hazard_construction",
            "retuned_during_repair",
        ]
    ]
)

,model_id,family,horizon_months,dataset_variant,training_rows,training_participants,improvement_event_rows,progression_event_rows,improvement_alpha,progression_alpha,continuous_features,categorical_features,endpoint_construction,baseline_hazard_construction,retuned_during_repair
0,B3,baseline,12,primary,2447,2447,65,89,10.0,10.0,age_at_landmark|education_years|index_mmse,sex,frozen Phase 3 cause-specific first-transition...,Breslow cause-specific hazard jumps combined t...,False
1,B3,baseline,36,primary,2447,2447,219,302,10.0,10.0,age_at_landmark|education_years|index_mmse,sex,frozen Phase 3 cause-specific first-transition...,Breslow cause-specific hazard jumps combined t...,False
2,D0,dynamic,12,primary,4517,1412,154,303,10.0,100.0,age_at_landmark|education_years|apoe_e4_dose|i...,sex,frozen Phase 3 cause-specific first-transition...,Breslow cause-specific hazard jumps combined t...,False
3,D0,dynamic,36,primary,4517,1412,503,1020,100.0,100.0,age_at_landmark|education_years|apoe_e4_dose|i...,sex,frozen Phase 3 cause-specific first-transition...,Breslow cause-specific hazard jumps combined t...,False
4,D3,dynamic,12,strictly_lagged_mmse,3134,1016,91,217,10.0,100.0,age_at_landmark|education_years|apoe_e4_dose|p...,sex,frozen Phase 3 cause-specific first-transition...,Breslow cause-specific hazard jumps combined t...,False
5,D3,dynamic,36,strictly_lagged_mmse,3134,1016,290,725,100.0,100.0,age_at_landmark|education_years|apoe_e4_dose|p...,sex,frozen Phase 3 cause-specific first-transition...,Breslow cause-specific hazard jumps combined t...,False


## Per-model software versions

In [7]:
software = pd.read_csv(LOCK / "model_software_versions.csv")
display(software.reset_index(drop=True))

,package_version_id,model_version_id,model_id,horizon_months,model_origin,training_python,training_platform,training_numpy,training_pandas,training_scipy,training_scikit_learn,training_statsmodels,package_joblib,version_source,source_scope
0,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-20260724-053b877edf3a,B1,12,verified_copy_from_original_lock,"3.10.20 (main, Jul 18 2026, 17:02:35) [Clang 2...",Linux-6.17.0-35-generic-x86_64-with-glibc2.39,2.2.6,2.3.3,1.15.3,1.7.2,0.14.6,1.5.3,results/csvs/phase3/phase3_run_metadata.json,original Phase 3 training environment; joblib ...
1,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-20260724-053b877edf3a,B1,24,verified_copy_from_original_lock,"3.10.20 (main, Jul 18 2026, 17:02:35) [Clang 2...",Linux-6.17.0-35-generic-x86_64-with-glibc2.39,2.2.6,2.3.3,1.15.3,1.7.2,0.14.6,1.5.3,results/csvs/phase3/phase3_run_metadata.json,original Phase 3 training environment; joblib ...
2,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-20260724-053b877edf3a,B1,36,verified_copy_from_original_lock,"3.10.20 (main, Jul 18 2026, 17:02:35) [Clang 2...",Linux-6.17.0-35-generic-x86_64-with-glibc2.39,2.2.6,2.3.3,1.15.3,1.7.2,0.14.6,1.5.3,results/csvs/phase3/phase3_run_metadata.json,original Phase 3 training environment; joblib ...
3,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-20260724-053b877edf3a,B1,60,verified_copy_from_original_lock,"3.10.20 (main, Jul 18 2026, 17:02:35) [Clang 2...",Linux-6.17.0-35-generic-x86_64-with-glibc2.39,2.2.6,2.3.3,1.15.3,1.7.2,0.14.6,1.5.3,results/csvs/phase3/phase3_run_metadata.json,original Phase 3 training environment; joblib ...
4,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-lockrepair-v2-294971626abc,B3,12,adni_only_outcome_blind_lock_repair,3.10.20,Linux-6.17.0-35-generic-x86_64-with-glibc2.39,2.2.6,2.3.3,1.15.3,1.7.2,0.14.6,1.5.3,models/phase3/phase4_lock_v2/repair_software_e...,ADNI-only repaired-model training environment
5,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-lockrepair-v2-294971626abc,B3,36,adni_only_outcome_blind_lock_repair,3.10.20,Linux-6.17.0-35-generic-x86_64-with-glibc2.39,2.2.6,2.3.3,1.15.3,1.7.2,0.14.6,1.5.3,models/phase3/phase4_lock_v2/repair_software_e...,ADNI-only repaired-model training environment
6,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-lockrepair-v2-294971626abc,D0,12,adni_only_outcome_blind_lock_repair,3.10.20,Linux-6.17.0-35-generic-x86_64-with-glibc2.39,2.2.6,2.3.3,1.15.3,1.7.2,0.14.6,1.5.3,models/phase3/phase4_lock_v2/repair_software_e...,ADNI-only repaired-model training environment
7,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-lockrepair-v2-294971626abc,D0,36,adni_only_outcome_blind_lock_repair,3.10.20,Linux-6.17.0-35-generic-x86_64-with-glibc2.39,2.2.6,2.3.3,1.15.3,1.7.2,0.14.6,1.5.3,models/phase3/phase4_lock_v2/repair_software_e...,ADNI-only repaired-model training environment
8,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-20260724-053b877edf3a,D1,12,verified_copy_from_original_lock,"3.10.20 (main, Jul 18 2026, 17:02:35) [Clang 2...",Linux-6.17.0-35-generic-x86_64-with-glibc2.39,2.2.6,2.3.3,1.15.3,1.7.2,0.14.6,1.5.3,results/csvs/phase3/phase3_run_metadata.json,original Phase 3 training environment; joblib ...
9,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-20260724-053b877edf3a,D1,24,verified_copy_from_original_lock,"3.10.20 (main, Jul 18 2026, 17:02:35) [Clang 2...",Linux-6.17.0-35-generic-x86_64-with-glibc2.39,2.2.6,2.3.3,1.15.3,1.7.2,0.14.6,1.5.3,results/csvs/phase3/phase3_run_metadata.json,original Phase 3 training environment; joblib ...


## Original versus repaired B1/D1 equivalence

In [8]:
equivalence = pd.read_csv(
    TABLES / "independent_b1_d1_equivalence.csv"
)
display(equivalence)
print(
    "All artifact hashes and predictions identical:",
    equivalence["status"].eq("PASS").all(),
)

,model_id,horizon_months,fixture_outer_fold,fixture_rows,original_sha256,repaired_sha256,artifact_hash_identical,maximum_probability_difference,metadata_exact,status
0,B1,12,0,20,b408adbf1644174870ad96a4df3df633e9a3f53c6b2a5a...,b408adbf1644174870ad96a4df3df633e9a3f53c6b2a5a...,True,0.0,True,PASS
1,B1,24,0,20,5cd4e5e45316f20bee3a3c0d88b11d9bf11b745f20bc46...,5cd4e5e45316f20bee3a3c0d88b11d9bf11b745f20bc46...,True,0.0,True,PASS
2,B1,36,0,20,52baf11e3cd593770ce25145a08fff317264aa0f95e972...,52baf11e3cd593770ce25145a08fff317264aa0f95e972...,True,0.0,True,PASS
3,B1,60,0,20,e3d9f777e6cdaacc45e7ae91f1273de08f97e6879cd35f...,e3d9f777e6cdaacc45e7ae91f1273de08f97e6879cd35f...,True,0.0,True,PASS
4,D1,12,0,20,f2e05fbfe91fdfe31ee46dcd674c09ffa46be7262ab299...,f2e05fbfe91fdfe31ee46dcd674c09ffa46be7262ab299...,True,0.0,True,PASS
5,D1,24,0,20,14cd2e62ca7ee94e0cdb749d1ecab981c2d14e643428b2...,14cd2e62ca7ee94e0cdb749d1ecab981c2d14e643428b2...,True,0.0,True,PASS
6,D1,36,0,20,6c867846fe5c295b070c8de8c9103abee9d907eb13fb72...,6c867846fe5c295b070c8de8c9103abee9d907eb13fb72...,True,0.0,True,PASS
7,D1,60,0,20,81b7653bd64f7770a72da7d3ad30f81becf4c797947f66...,81b7653bd64f7770a72da7d3ad30f81becf4c797947f66...,True,0.0,True,PASS


All artifact hashes and predictions identical: True


## Independent D0/B3/D3 deterministic inference

In [9]:
fixtures = pd.read_csv(
    TABLES / "independent_inference_fixture_verification.csv"
)
display(fixtures)
print(
    "All held-out ADNI fixtures reproduced:",
    fixtures["status"].eq("PASS").all(),
)

,model_id,horizon_months,rows,maximum_expected_difference,maximum_repeat_difference,metadata_exact,probabilities_bounded_and_sum_one,predictor_only_fixture,status
0,B3,12,20,0.0,0.0,True,True,True,PASS
1,B3,36,20,0.0,0.0,True,True,True,PASS
2,D0,12,20,0.0,0.0,True,True,True,PASS
3,D0,36,20,0.0,0.0,True,True,True,PASS
4,D3,12,20,0.0,0.0,True,True,True,PASS
5,D3,36,20,0.0,0.0,True,True,True,PASS


All held-out ADNI fixtures reproduced: True


## D3 strictly pre-landmark MMSE verification

In [10]:
d3_temporal = pd.read_csv(
    TABLES / "independent_d3_temporal_verification.csv"
)
display(d3_temporal)
print(
    "No same-day/post-landmark MMSE used:",
    d3_temporal["status"].eq("PASS").all(),
)

,model_id,horizon_months,rows,participants,previous_mmse_missing_rows,previous_mmse_not_strictly_before_landmark_rows,nonpositive_time_since_previous_mmse_rows,strict_availability_flag_false_rows,current_or_same_day_mmse_in_model_features,status
0,D3,12,3134,1016,0,0,0,0,False,PASS
1,D3,36,3134,1016,0,0,0,0,False,PASS


No same-day/post-landmark MMSE used: True


## Dynamic participant-grouping verification

In [11]:
grouping = pd.read_csv(
    TABLES / "independent_participant_grouping_verification.csv"
)
display(grouping)
print(
    "Every dynamic grouping check passed:",
    grouping["status"].eq("PASS").all(),
)

,scope,participants,mismatches_or_multiple_roles,status
0,outer_assignments,2447,0,PASS
1,nested_assignments,2447,0,PASS
2,D0_12m,1412,0,PASS
3,D0_36m,1412,0,PASS
4,D1_12m,1412,0,PASS
5,D1_24m,1412,0,PASS
6,D1_36m,1412,0,PASS
7,D1_60m,1412,0,PASS
8,D3_12m,1016,0,PASS
9,D3_36m,1016,0,PASS


Every dynamic grouping check passed: True


## Preprocessing, features, coefficients, and baseline hazards

In [12]:
preprocessing = pd.read_csv(LOCK / "frozen_preprocessing_parameters.csv")
schema = pd.read_csv(LOCK / "frozen_feature_schema.csv")
coefficients = pd.read_csv(LOCK / "frozen_model_coefficients.csv")
hazards = pd.read_csv(LOCK / "frozen_baseline_hazards.csv")

coverage_tables = pd.DataFrame(
    {
        "table": [
            "frozen_feature_schema",
            "frozen_preprocessing_parameters",
            "frozen_model_coefficients",
            "frozen_baseline_hazards",
        ],
        "rows": [
            len(schema), len(preprocessing), len(coefficients), len(hazards)
        ],
        "models": [
            schema["model_id"].nunique(),
            preprocessing["model_id"].nunique(),
            coefficients["model_id"].nunique(),
            hazards["model_id"].nunique(),
        ],
        "model_horizon_pairs": [
            schema[["model_id", "horizon_months"]].drop_duplicates().shape[0],
            preprocessing[["model_id", "horizon_months"]].drop_duplicates().shape[0],
            coefficients[["model_id", "horizon_months"]].drop_duplicates().shape[0],
            hazards[["model_id", "horizon_months"]].drop_duplicates().shape[0],
        ],
    }
)
display(coverage_tables)
display(schema.sort_values(["model_id", "horizon_months"]).reset_index(drop=True))

,table,rows,models,model_horizon_pairs
0,frozen_feature_schema,82,5,14
1,frozen_preprocessing_parameters,82,5,14
2,frozen_model_coefficients,192,5,14
3,frozen_baseline_hazards,6048,5,14


,package_version_id,model_version_id,model_origin,model_id,family,horizon_months,feature,data_type,required_column,missing_value_handling,adni_observed_min,adni_observed_max,allowed_levels,external_transform_rule
0,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-20260724-053b877edf3a,verified_copy_from_original_lock,B1,baseline,12,age_at_landmark,continuous,True,ADNI final-training median imputation,53.804244,94.713210,NaN,apply frozen ADNI preprocessor unchanged
1,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-20260724-053b877edf3a,verified_copy_from_original_lock,B1,baseline,12,education_years,continuous,True,ADNI final-training median imputation,3.000000,20.000000,NaN,apply frozen ADNI preprocessor unchanged
2,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-20260724-053b877edf3a,verified_copy_from_original_lock,B1,baseline,12,apoe_e4_dose,continuous,True,ADNI final-training median imputation,0.000000,2.000000,NaN,apply frozen ADNI preprocessor unchanged
3,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-20260724-053b877edf3a,verified_copy_from_original_lock,B1,baseline,12,index_mmse,continuous,True,ADNI final-training median imputation,13.000000,30.000000,NaN,apply frozen ADNI preprocessor unchanged
4,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-20260724-053b877edf3a,verified_copy_from_original_lock,B1,baseline,12,sex,categorical,True,ADNI final-training most-frequent imputation; ...,NaN,NaN,Female|Male,apply frozen ADNI preprocessor unchanged
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-lockrepair-v2-294971626abc,adni_only_outcome_blind_lock_repair,D3,dynamic,36,apoe_e4_dose,continuous,True,ADNI final-training median imputation,0.000000,2.000000,NaN,apply frozen ADNI preprocessor unchanged
78,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-lockrepair-v2-294971626abc,adni_only_outcome_blind_lock_repair,D3,dynamic,36,previous_mmse,continuous,True,ADNI final-training median imputation,10.000000,30.000000,NaN,apply frozen ADNI preprocessor unchanged
79,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-lockrepair-v2-294971626abc,adni_only_outcome_blind_lock_repair,D3,dynamic,36,time_since_previous_mmse_years,continuous,True,ADNI final-training median imputation,0.002738,9.486653,NaN,apply frozen ADNI preprocessor unchanged
80,phase3-adni-lockrepair-v2-294971626abc,phase3-adni-lockrepair-v2-294971626abc,adni_only_outcome_blind_lock_repair,D3,dynamic,36,elapsed_years_from_index,continuous,True,ADNI final-training median imputation,0.000000,17.327858,NaN,apply frozen ADNI preprocessor unchanged


## Original and repaired checksum verification

In [13]:
original_checks = pd.read_csv(
    TABLES / "independent_original_lock_verification.csv"
)
package_checks = pd.read_csv(
    TABLES / "independent_package_checksum_verification.csv"
)
display(
    pd.DataFrame(
        [
            {
                "scope": "original immutable lock",
                "files_checked": len(original_checks),
                "failed": int(original_checks["status"].ne("PASS").sum()),
                "status": (
                    "PASS"
                    if original_checks["status"].eq("PASS").all()
                    else "FAIL"
                ),
            },
            {
                "scope": "repaired v2 lock",
                "files_checked": len(package_checks),
                "failed": int(package_checks["status"].ne("PASS").sum()),
                "status": (
                    "PASS"
                    if package_checks["status"].eq("PASS").all()
                    else "FAIL"
                ),
            },
        ]
    )
)

,scope,files_checked,failed,status
0,original immutable lock,16,0,PASS
1,repaired v2 lock,55,0,PASS


## Outcome-blind access audit

In [14]:
access = pd.read_csv(LOCK / "access_audit.csv")
forbidden_fragments = [
    "/processeddata/oasis3/",
    "/datasets/oasis3/",
    "/datasets/oasis/",
    "/results/csvs/phase1/oasis3/",
    "/results/csvs/phase2/oasis3/",
]
paths = "/" + access["path"].astype(str).str.lower()
forbidden = paths.map(
    lambda value: any(fragment in value for fragment in forbidden_fragments)
)
display(
    access.groupby("access_class", dropna=False)
    .size()
    .rename("records")
    .reset_index()
)
print(f"Input-read records: {len(access)}")
print(f"Forbidden OASIS participant-file paths: {int(forbidden.sum())}")
print(
    "OASIS participant-level features accessed:",
    access["oasis_participant_level_features_accessed"].astype(bool).any(),
)
print(
    "OASIS participant-level outcomes accessed:",
    access["oasis_participant_level_outcomes_accessed"].astype(bool).any(),
)

,access_class,records
0,adni_record_level,17
1,frozen_reference,11
2,immutable_original_lock,3
3,immutable_original_model,8
4,phase3_checksum_metadata,1
5,phase3_machine_readable_result,4
6,phase3_provenance,1
7,source_code,3


Input-read records: 48
Forbidden OASIS participant-file paths: 0
OASIS participant-level features accessed: False
OASIS participant-level outcomes accessed: False


## Independent QC checklist

In [15]:
qc = pd.read_csv(TABLES / "independent_qc_results.csv")
display(qc)
print(f"Critical checks: {int(qc['critical'].sum())}")
print(
    "Critical failures:",
    int((qc["critical"] & qc["status"].ne("PASS")).sum()),
)

,section,check,critical,status,details
0,package,required self-contained support files exist,True,PASS,required=31; missing=[]
1,original_lock,all original-lock checksums and sizes still pass,True,PASS,rows=16; failed=0
2,package,external manifest authenticates model index,True,PASS,observed=6fdae9393bb478f78d6c534416516023a5e5d...
3,package,external manifest authenticates checksum manifest,True,PASS,observed=e84bd8bfafbdb755b45bfc8fa3e930bd06bc0...
4,package,checksum manifest covers every in-scope packag...,True,PASS,manifest=55; actual_in_scope=55; missing=[]; e...
5,package,all repaired-package checksums and sizes pass,True,PASS,rows=55; failed=0
6,inventory,every Phase 4 required model and horizon is pr...,True,PASS,"present=[('B1', 12), ('B1', 24), ('B1', 36), (..."
7,inventory,frozen index and inventory contain the same 14...,True,PASS,index_rows=14; inventory_rows=14
8,outcome_blindness,frozen index certifies no external participant...,True,PASS,development=ADNI; features_accessed=False; out...
9,artifacts,all 14 model bundles contain complete frozen p...,True,PASS,models=14; failed=0


Critical checks: 18
Critical failures: 0


## Reports and Phase 4 restart handoff

In [16]:
for report_name in [
    "phase3_outcome_blind_lock_repair_report.md",
    "phase3_lock_repair_inference_compatibility_report.md",
    "phase3_lock_repair_independent_qc_report.md",
    "phase4_restart_handoff.md",
]:
    display(Markdown((REPORTS / report_name).read_text()))

# Outcome-Blind Phase 3 Lock Repair

Generated: 2026-07-29T02:33:23.848356+00:00

## Package

New lock: `models/phase3/phase4_lock_v2`

Package version: `phase3-adni-lockrepair-v2-294971626abc`

The original lock was verified before and after repair and was not modified.
B1 and D1 were copied byte-for-byte and were not refit. D0, B3, and D3 were
fitted on ADNI only using the exact Phase 3 model family, predictor definitions,
endpoint construction, preprocessing, participant-grouped tuning decisions,
ridge penalty recovery rule, and Breslow baseline-hazard construction.

## Recovered penalties

| model_id   |   horizon_months | cause       |   recovered_final_alpha | outer_fold_selected_alphas    | retuned_during_repair   |
|:-----------|-----------------:|:------------|------------------------:|:------------------------------|:------------------------|
| D0         |               12 | improvement |                      10 | 0:100|1:10|2:100|3:10|4:10    | False                   |
| D0         |               12 | progression |                     100 | 0:10|1:100|2:100|3:100|4:100  | False                   |
| D0         |               36 | improvement |                     100 | 0:100|1:100|2:100|3:100|4:100 | False                   |
| D0         |               36 | progression |                     100 | 0:100|1:100|2:100|3:100|4:100 | False                   |
| B3         |               12 | improvement |                      10 | 0:100|1:10|2:10|3:10|4:10     | False                   |
| B3         |               12 | progression |                      10 | 0:10|1:10|2:10|3:10|4:10      | False                   |
| B3         |               36 | improvement |                      10 | 0:10|1:10|2:10|3:10|4:10      | False                   |
| B3         |               36 | progression |                      10 | 0:10|1:100|2:10|3:10|4:100    | False                   |
| D3         |               12 | improvement |                      10 | 0:10|1:10|2:10|3:10|4:10      | False                   |
| D3         |               12 | progression |                     100 | 0:10|1:100|2:100|3:100|4:100  | False                   |
| D3         |               36 | improvement |                     100 | 0:100|1:100|2:100|3:10|4:100  | False                   |
| D3         |               36 | progression |                     100 | 0:100|1:100|2:100|3:100|4:100 | False                   |

## Repaired model training metadata

| model_id   |   horizon_months |   training_rows |   training_participants |   improvement_event_rows |   progression_event_rows |   improvement_alpha |   progression_alpha | continuous_features                                                                                                |
|:-----------|-----------------:|----------------:|------------------------:|-------------------------:|-------------------------:|--------------------:|--------------------:|:-------------------------------------------------------------------------------------------------------------------|
| B3         |               12 |            2447 |                    2447 |                       65 |                       89 |                  10 |                  10 | age_at_landmark|education_years|index_mmse                                                                         |
| B3         |               36 |            2447 |                    2447 |                      219 |                      302 |                  10 |                  10 | age_at_landmark|education_years|index_mmse                                                                         |
| D0         |               12 |            4517 |                    1412 |                      154 |                      303 |                  10 |                 100 | age_at_landmark|education_years|apoe_e4_dose|index_mmse|elapsed_years_from_index                                   |
| D0         |               36 |            4517 |                    1412 |                      503 |                     1020 |                 100 |                 100 | age_at_landmark|education_years|apoe_e4_dose|index_mmse|elapsed_years_from_index                                   |
| D3         |               12 |            3134 |                    1016 |                       91 |                      217 |                  10 |                 100 | age_at_landmark|education_years|apoe_e4_dose|previous_mmse|time_since_previous_mmse_years|elapsed_years_from_index |
| D3         |               36 |            3134 |                    1016 |                      290 |                      725 |                 100 |                 100 | age_at_landmark|education_years|apoe_e4_dose|previous_mmse|time_since_previous_mmse_years|elapsed_years_from_index |

## Original B1/D1 equivalence

| model_id   |   horizon_months | artifact_hash_identical   |   maximum_probability_difference | metadata_exact   | status   |
|:-----------|-----------------:|:--------------------------|---------------------------------:|:-----------------|:---------|
| B1         |               12 | True                      |                                0 | True             | PASS     |
| B1         |               36 | True                      |                                0 | True             | PASS     |
| B1         |               24 | True                      |                                0 | True             | PASS     |
| B1         |               60 | True                      |                                0 | True             | PASS     |
| D1         |               12 | True                      |                                0 | True             | PASS     |
| D1         |               36 | True                      |                                0 | True             | PASS     |
| D1         |               24 | True                      |                                0 | True             | PASS     |
| D1         |               60 | True                      |                                0 | True             | PASS     |

## New-model deterministic fixtures

| model_id   |   horizon_months | fixture_source           |   fixture_rows |   maximum_repeat_prediction_difference | status   |
|:-----------|-----------------:|:-------------------------|---------------:|---------------------------------------:|:---------|
| D0         |               12 | ADNI frozen outer fold 0 |             20 |                                      0 | PASS     |
| D0         |               36 | ADNI frozen outer fold 0 |             20 |                                      0 | PASS     |
| B3         |               12 | ADNI frozen outer fold 0 |             20 |                                      0 | PASS     |
| B3         |               36 | ADNI frozen outer fold 0 |             20 |                                      0 | PASS     |
| D3         |               12 | ADNI frozen outer fold 0 |             20 |                                      0 | PASS     |
| D3         |               36 | ADNI frozen outer fold 0 |             20 |                                      0 | PASS     |

## D3 temporal protection

| model_id   |   horizon_months |   rows |   participants |   previous_mmse_missing_rows |   previous_mmse_not_strictly_before_landmark_rows |   nonpositive_time_since_previous_mmse_rows |   strict_availability_flag_false_rows | current_or_same_day_mmse_in_model_features   | status   |
|:-----------|-----------------:|-------:|---------------:|-----------------------------:|--------------------------------------------------:|--------------------------------------------:|--------------------------------------:|:---------------------------------------------|:---------|
| D3         |               12 |   3134 |           1016 |                            0 |                                                 0 |                                           0 |                                     0 | False                                        | PASS     |
| D3         |               36 |   3134 |           1016 |                            0 |                                                 0 |                                           0 |                                     0 | False                                        | PASS     |

## Participant grouping

| scope        | check                                                | status   | details                                                                |
|:-------------|:-----------------------------------------------------|:---------|:-----------------------------------------------------------------------|
| outer_folds  | one outer fold per ADNI participant                  | PASS     | participants=2447; maximum_folds=1                                     |
| nested_folds | one nested role per participant and outer split      | PASS     | rows=12235; maximum_roles=1                                            |
| D0_12m       | dynamic rows retain frozen participant-grouped folds | PASS     | rows=4517; participants=1412; fold_mismatches=0; saved_penalty_rows=10 |
| D0_36m       | dynamic rows retain frozen participant-grouped folds | PASS     | rows=4517; participants=1412; fold_mismatches=0; saved_penalty_rows=10 |
| D3_12m       | dynamic rows retain frozen participant-grouped folds | PASS     | rows=3134; participants=1016; fold_mismatches=0; saved_penalty_rows=10 |
| D3_36m       | dynamic rows retain frozen participant-grouped folds | PASS     | rows=3134; participants=1016; fold_mismatches=0; saved_penalty_rows=10 |

## Per-model software versions

| model_id   |   horizon_months | model_origin                        | training_python                                       | training_numpy   | training_pandas   | training_scipy   | training_scikit_learn   | training_statsmodels   | package_joblib   | version_source                                                |
|:-----------|-----------------:|:------------------------------------|:------------------------------------------------------|:-----------------|:------------------|:-----------------|:------------------------|:-----------------------|:-----------------|:--------------------------------------------------------------|
| B1         |               12 | verified_copy_from_original_lock    | 3.10.20 (main, Jul 18 2026, 17:02:35) [Clang 22.1.3 ] | 2.2.6            | 2.3.3             | 1.15.3           | 1.7.2                   | 0.14.6                 | 1.5.3            | results/csvs/phase3/phase3_run_metadata.json                  |
| B1         |               24 | verified_copy_from_original_lock    | 3.10.20 (main, Jul 18 2026, 17:02:35) [Clang 22.1.3 ] | 2.2.6            | 2.3.3             | 1.15.3           | 1.7.2                   | 0.14.6                 | 1.5.3            | results/csvs/phase3/phase3_run_metadata.json                  |
| B1         |               36 | verified_copy_from_original_lock    | 3.10.20 (main, Jul 18 2026, 17:02:35) [Clang 22.1.3 ] | 2.2.6            | 2.3.3             | 1.15.3           | 1.7.2                   | 0.14.6                 | 1.5.3            | results/csvs/phase3/phase3_run_metadata.json                  |
| B1         |               60 | verified_copy_from_original_lock    | 3.10.20 (main, Jul 18 2026, 17:02:35) [Clang 22.1.3 ] | 2.2.6            | 2.3.3             | 1.15.3           | 1.7.2                   | 0.14.6                 | 1.5.3            | results/csvs/phase3/phase3_run_metadata.json                  |
| B3         |               12 | adni_only_outcome_blind_lock_repair | 3.10.20                                               | 2.2.6            | 2.3.3             | 1.15.3           | 1.7.2                   | 0.14.6                 | 1.5.3            | models/phase3/phase4_lock_v2/repair_software_environment.json |
| B3         |               36 | adni_only_outcome_blind_lock_repair | 3.10.20                                               | 2.2.6            | 2.3.3             | 1.15.3           | 1.7.2                   | 0.14.6                 | 1.5.3            | models/phase3/phase4_lock_v2/repair_software_environment.json |
| D0         |               12 | adni_only_outcome_blind_lock_repair | 3.10.20                                               | 2.2.6            | 2.3.3             | 1.15.3           | 1.7.2                   | 0.14.6                 | 1.5.3            | models/phase3/phase4_lock_v2/repair_software_environment.json |
| D0         |               36 | adni_only_outcome_blind_lock_repair | 3.10.20                                               | 2.2.6            | 2.3.3             | 1.15.3           | 1.7.2                   | 0.14.6                 | 1.5.3            | models/phase3/phase4_lock_v2/repair_software_environment.json |
| D1         |               12 | verified_copy_from_original_lock    | 3.10.20 (main, Jul 18 2026, 17:02:35) [Clang 22.1.3 ] | 2.2.6            | 2.3.3             | 1.15.3           | 1.7.2                   | 0.14.6                 | 1.5.3            | results/csvs/phase3/phase3_run_metadata.json                  |
| D1         |               24 | verified_copy_from_original_lock    | 3.10.20 (main, Jul 18 2026, 17:02:35) [Clang 22.1.3 ] | 2.2.6            | 2.3.3             | 1.15.3           | 1.7.2                   | 0.14.6                 | 1.5.3            | results/csvs/phase3/phase3_run_metadata.json                  |
| D1         |               36 | verified_copy_from_original_lock    | 3.10.20 (main, Jul 18 2026, 17:02:35) [Clang 22.1.3 ] | 2.2.6            | 2.3.3             | 1.15.3           | 1.7.2                   | 0.14.6                 | 1.5.3            | results/csvs/phase3/phase3_run_metadata.json                  |
| D1         |               60 | verified_copy_from_original_lock    | 3.10.20 (main, Jul 18 2026, 17:02:35) [Clang 22.1.3 ] | 2.2.6            | 2.3.3             | 1.15.3           | 1.7.2                   | 0.14.6                 | 1.5.3            | results/csvs/phase3/phase3_run_metadata.json                  |
| D3         |               12 | adni_only_outcome_blind_lock_repair | 3.10.20                                               | 2.2.6            | 2.3.3             | 1.15.3           | 1.7.2                   | 0.14.6                 | 1.5.3            | models/phase3/phase4_lock_v2/repair_software_environment.json |
| D3         |               36 | adni_only_outcome_blind_lock_repair | 3.10.20                                               | 2.2.6            | 2.3.3             | 1.15.3           | 1.7.2                   | 0.14.6                 | 1.5.3            | models/phase3/phase4_lock_v2/repair_software_environment.json |

## External-cohort protection

The access audit contains 48 input-read records. Every record
certifies that OASIS-3 participant-level features and outcomes were not
accessed. Phase 4 was not resumed.


# Phase 3 Lock Repair Inference Compatibility

Generated: 2026-07-29T02:33:23.848356+00:00

The existing generic `phase3_inference.py` required no change. It reads each
bundle's frozen continuous and categorical feature lists, applies the serialized
preprocessor, verifies transformed feature order, and combines paired
cause-specific hazards into coherent probabilities.

## B1/D1 equivalence

| model_id   |   horizon_months | comparison                |   fixture_outer_fold |   fixture_rows | original_sha256                                                  | repaired_sha256                                                  | artifact_hash_identical   |   maximum_probability_difference | metadata_exact   | status   |
|:-----------|-----------------:|:--------------------------|---------------------:|---------------:|:-----------------------------------------------------------------|:-----------------------------------------------------------------|:--------------------------|---------------------------------:|:-----------------|:---------|
| B1         |               12 | original_vs_verified_copy |                    0 |             20 | b408adbf1644174870ad96a4df3df633e9a3f53c6b2a5aa727ea1c1725dbd052 | b408adbf1644174870ad96a4df3df633e9a3f53c6b2a5aa727ea1c1725dbd052 | True                      |                                0 | True             | PASS     |
| B1         |               36 | original_vs_verified_copy |                    0 |             20 | 52baf11e3cd593770ce25145a08fff317264aa0f95e97250686724cfff021265 | 52baf11e3cd593770ce25145a08fff317264aa0f95e97250686724cfff021265 | True                      |                                0 | True             | PASS     |
| B1         |               24 | original_vs_verified_copy |                    0 |             20 | 5cd4e5e45316f20bee3a3c0d88b11d9bf11b745f20bc464b430f20264d9bc164 | 5cd4e5e45316f20bee3a3c0d88b11d9bf11b745f20bc464b430f20264d9bc164 | True                      |                                0 | True             | PASS     |
| B1         |               60 | original_vs_verified_copy |                    0 |             20 | e3d9f777e6cdaacc45e7ae91f1273de08f97e6879cd35fd8ae7b5a88d2f77144 | e3d9f777e6cdaacc45e7ae91f1273de08f97e6879cd35fd8ae7b5a88d2f77144 | True                      |                                0 | True             | PASS     |
| D1         |               12 | original_vs_verified_copy |                    0 |             20 | f2e05fbfe91fdfe31ee46dcd674c09ffa46be7262ab299481f6f72f26e355c99 | f2e05fbfe91fdfe31ee46dcd674c09ffa46be7262ab299481f6f72f26e355c99 | True                      |                                0 | True             | PASS     |
| D1         |               36 | original_vs_verified_copy |                    0 |             20 | 6c867846fe5c295b070c8de8c9103abee9d907eb13fb7205f3690a9315843534 | 6c867846fe5c295b070c8de8c9103abee9d907eb13fb7205f3690a9315843534 | True                      |                                0 | True             | PASS     |
| D1         |               24 | original_vs_verified_copy |                    0 |             20 | 14cd2e62ca7ee94e0cdb749d1ecab981c2d14e643428b2cc6faaca123b7a96f3 | 14cd2e62ca7ee94e0cdb749d1ecab981c2d14e643428b2cc6faaca123b7a96f3 | True                      |                                0 | True             | PASS     |
| D1         |               60 | original_vs_verified_copy |                    0 |             20 | 81b7653bd64f7770a72da7d3ad30f81becf4c797947f66b366fc6fb932602d77 | 81b7653bd64f7770a72da7d3ad30f81becf4c797947f66b366fc6fb932602d77 | True                      |                                0 | True             | PASS     |

## D0/B3/D3 fixtures

| model_id   |   horizon_months | artifact_path                                                                    | input_path                | expected_path                | fixture_source           |   fixture_rows | outcome_columns_present   |   maximum_repeat_prediction_difference | status   |
|:-----------|-----------------:|:---------------------------------------------------------------------------------|:--------------------------|:-----------------------------|:-------------------------|---------------:|:--------------------------|---------------------------------------:|:---------|
| D0         |               12 | models/phase3/phase4_lock_v2/artifacts/frozen_adni_d0_12m_ridge_competing.joblib | fixtures/d0_12m_input.csv | fixtures/d0_12m_expected.csv | ADNI frozen outer fold 0 |             20 | False                     |                                      0 | PASS     |
| D0         |               36 | models/phase3/phase4_lock_v2/artifacts/frozen_adni_d0_36m_ridge_competing.joblib | fixtures/d0_36m_input.csv | fixtures/d0_36m_expected.csv | ADNI frozen outer fold 0 |             20 | False                     |                                      0 | PASS     |
| B3         |               12 | models/phase3/phase4_lock_v2/artifacts/frozen_adni_b3_12m_ridge_competing.joblib | fixtures/b3_12m_input.csv | fixtures/b3_12m_expected.csv | ADNI frozen outer fold 0 |             20 | False                     |                                      0 | PASS     |
| B3         |               36 | models/phase3/phase4_lock_v2/artifacts/frozen_adni_b3_36m_ridge_competing.joblib | fixtures/b3_36m_input.csv | fixtures/b3_36m_expected.csv | ADNI frozen outer fold 0 |             20 | False                     |                                      0 | PASS     |
| D3         |               12 | models/phase3/phase4_lock_v2/artifacts/frozen_adni_d3_12m_ridge_competing.joblib | fixtures/d3_12m_input.csv | fixtures/d3_12m_expected.csv | ADNI frozen outer fold 0 |             20 | False                     |                                      0 | PASS     |
| D3         |               36 | models/phase3/phase4_lock_v2/artifacts/frozen_adni_d3_36m_ridge_competing.joblib | fixtures/d3_36m_input.csv | fixtures/d3_36m_expected.csv | ADNI frozen outer fold 0 |             20 | False                     |                                      0 | PASS     |

All fixture inputs come from frozen ADNI outer fold
0. They contain predictor columns
only; no outcome column is saved in the package fixtures.


# Independent QC — Phase 3 Lock Repair

Generated: 2026-07-29T02:37:21.308871+00:00

Verifier: `scripts/phase3_lock_repair_qc.py`

The verifier is independent of the repair builder: it does not import the
builder, does not refit any model, and does not access OASIS-3 participant-level
features or outcomes.

## Decision

**RESUME PHASE 4 WITH RESTRICTIONS**

Phase 4 must use: `models/phase3/phase4_lock_v2`

## QC checklist

| section           | check                                                                    | critical   | status   | details                                                                                                                                                                                                     |
|:------------------|:-------------------------------------------------------------------------|:-----------|:---------|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| package           | required self-contained support files exist                              | True       | PASS     | required=31; missing=[]                                                                                                                                                                                     |
| original_lock     | all original-lock checksums and sizes still pass                         | True       | PASS     | rows=16; failed=0                                                                                                                                                                                           |
| package           | external manifest authenticates model index                              | True       | PASS     | observed=6fdae9393bb478f78d6c534416516023a5e5dfe20b0250cbb3b50b694a0ca782                                                                                                                                   |
| package           | external manifest authenticates checksum manifest                        | True       | PASS     | observed=e84bd8bfafbdb755b45bfc8fa3e930bd06bc04fec2bc3a331fdf243e2342630b                                                                                                                                   |
| package           | checksum manifest covers every in-scope package file                     | True       | PASS     | manifest=55; actual_in_scope=55; missing=[]; extra=[]                                                                                                                                                       |
| package           | all repaired-package checksums and sizes pass                            | True       | PASS     | rows=55; failed=0                                                                                                                                                                                           |
| inventory         | every Phase 4 required model and horizon is present                      | True       | PASS     | present=[('B1', 12), ('B1', 24), ('B1', 36), ('B1', 60), ('B3', 12), ('B3', 36), ('D0', 12), ('D0', 36), ('D1', 12), ('D1', 24), ('D1', 36), ('D1', 60), ('D3', 12), ('D3', 36)]; missing=[]; unexpected=[] |
| inventory         | frozen index and inventory contain the same 14 bundles                   | True       | PASS     | index_rows=14; inventory_rows=14                                                                                                                                                                            |
| outcome_blindness | frozen index certifies no external participant-level access              | True       | PASS     | development=ADNI; features_accessed=False; outcomes_accessed=False                                                                                                                                          |
| artifacts         | all 14 model bundles contain complete frozen prediction state            | True       | PASS     | models=14; failed=0                                                                                                                                                                                         |
| specification     | saved penalties reproduce the Phase 3 modal-fold rule without retuning   | True       | PASS     | decisions=12; failed=0                                                                                                                                                                                      |
| inference         | packaged inference is prediction-only                                    | True       | PASS     | no .fit( call found in the packaged inference entrypoint                                                                                                                                                    |
| inference         | D0/B3/D3 predictions independently reproduce on held-out ADNI rows       | True       | PASS     | fixtures=6; max_difference=0                                                                                                                                                                                |
| inference         | B1/D1 predictions are identical before and after repair                  | True       | PASS     | comparisons=8; max_difference=0                                                                                                                                                                             |
| temporal          | D3 uses only strictly pre-landmark MMSE                                  | True       | PASS     | rows=[3134, 3134]; non_strict=0                                                                                                                                                                             |
| grouping          | all dynamic processing retains frozen participant-grouped folds          | True       | PASS     | checks=10; failed=0                                                                                                                                                                                         |
| outcome_blindness | access audit contains no OASIS participant-level feature or outcome file | True       | PASS     | records=48; forbidden_paths=0; external_access_flags=0                                                                                                                                                      |
| workflow          | Phase 4 was not resumed by the repair                                    | True       | PASS     | new lock is staged for later use; verifier performed ADNI/package checks only and did not execute external validation                                                                                       |

## Complete model-artifact verification

| model_id   |   horizon_months | artifact_hash_and_size   | required_bundle_keys   | feature_order   | preprocessing   | coefficients   | baseline_hazard   | ridge_penalties   | training_counts   | software_versions   | adni_outcome_blind_certification   | copy_or_refit_rule   | status   |
|:-----------|-----------------:|:-------------------------|:-----------------------|:----------------|:----------------|:---------------|:------------------|:------------------|:------------------|:--------------------|:-----------------------------------|:---------------------|:---------|
| B1         |               12 | True                     | True                   | True            | True            | True           | True              | True              | True              | True                | True                               | True                 | PASS     |
| B1         |               24 | True                     | True                   | True            | True            | True           | True              | True              | True              | True                | True                               | True                 | PASS     |
| B1         |               36 | True                     | True                   | True            | True            | True           | True              | True              | True              | True                | True                               | True                 | PASS     |
| B1         |               60 | True                     | True                   | True            | True            | True           | True              | True              | True              | True                | True                               | True                 | PASS     |
| B3         |               12 | True                     | True                   | True            | True            | True           | True              | True              | True              | True                | True                               | True                 | PASS     |
| B3         |               36 | True                     | True                   | True            | True            | True           | True              | True              | True              | True                | True                               | True                 | PASS     |
| D0         |               12 | True                     | True                   | True            | True            | True           | True              | True              | True              | True                | True                               | True                 | PASS     |
| D0         |               36 | True                     | True                   | True            | True            | True           | True              | True              | True              | True                | True                               | True                 | PASS     |
| D1         |               12 | True                     | True                   | True            | True            | True           | True              | True              | True              | True                | True                               | True                 | PASS     |
| D1         |               24 | True                     | True                   | True            | True            | True           | True              | True              | True              | True                | True                               | True                 | PASS     |
| D1         |               36 | True                     | True                   | True            | True            | True           | True              | True              | True              | True                | True                               | True                 | PASS     |
| D1         |               60 | True                     | True                   | True            | True            | True           | True              | True              | True              | True                | True                               | True                 | PASS     |
| D3         |               12 | True                     | True                   | True            | True            | True           | True              | True              | True              | True                | True                               | True                 | PASS     |
| D3         |               36 | True                     | True                   | True            | True            | True           | True              | True              | True              | True                | True                               | True                 | PASS     |

## Independent D0/B3/D3 inference

| model_id   |   horizon_months |   rows |   maximum_expected_difference |   maximum_repeat_difference | metadata_exact   | probabilities_bounded_and_sum_one   | predictor_only_fixture   | status   |
|:-----------|-----------------:|-------:|------------------------------:|----------------------------:|:-----------------|:------------------------------------|:-------------------------|:---------|
| B3         |               12 |     20 |                             0 |                           0 | True             | True                                | True                     | PASS     |
| B3         |               36 |     20 |                             0 |                           0 | True             | True                                | True                     | PASS     |
| D0         |               12 |     20 |                             0 |                           0 | True             | True                                | True                     | PASS     |
| D0         |               36 |     20 |                             0 |                           0 | True             | True                                | True                     | PASS     |
| D3         |               12 |     20 |                             0 |                           0 | True             | True                                | True                     | PASS     |
| D3         |               36 |     20 |                             0 |                           0 | True             | True                                | True                     | PASS     |

## Independent B1/D1 equivalence

| model_id   |   horizon_months | artifact_hash_identical   |   maximum_probability_difference | metadata_exact   | status   |
|:-----------|-----------------:|:--------------------------|---------------------------------:|:-----------------|:---------|
| B1         |               12 | True                      |                                0 | True             | PASS     |
| B1         |               24 | True                      |                                0 | True             | PASS     |
| B1         |               36 | True                      |                                0 | True             | PASS     |
| B1         |               60 | True                      |                                0 | True             | PASS     |
| D1         |               12 | True                      |                                0 | True             | PASS     |
| D1         |               24 | True                      |                                0 | True             | PASS     |
| D1         |               36 | True                      |                                0 | True             | PASS     |
| D1         |               60 | True                      |                                0 | True             | PASS     |

## D3 strict temporal verification

| model_id   |   horizon_months |   rows |   participants |   previous_mmse_missing_rows |   previous_mmse_not_strictly_before_landmark_rows |   nonpositive_time_since_previous_mmse_rows |   strict_availability_flag_false_rows | current_or_same_day_mmse_in_model_features   | status   |
|:-----------|-----------------:|-------:|---------------:|-----------------------------:|--------------------------------------------------:|--------------------------------------------:|--------------------------------------:|:---------------------------------------------|:---------|
| D3         |               12 |   3134 |           1016 |                            0 |                                                 0 |                                           0 |                                     0 | False                                        | PASS     |
| D3         |               36 |   3134 |           1016 |                            0 |                                                 0 |                                           0 |                                     0 | False                                        | PASS     |

## Dynamic participant grouping

| scope              |   participants |   mismatches_or_multiple_roles | status   |
|:-------------------|---------------:|-------------------------------:|:---------|
| outer_assignments  |           2447 |                              0 | PASS     |
| nested_assignments |           2447 |                              0 | PASS     |
| D0_12m             |           1412 |                              0 | PASS     |
| D0_36m             |           1412 |                              0 | PASS     |
| D1_12m             |           1412 |                              0 | PASS     |
| D1_24m             |           1412 |                              0 | PASS     |
| D1_36m             |           1412 |                              0 | PASS     |
| D1_60m             |           1412 |                              0 | PASS     |
| D3_12m             |           1016 |                              0 | PASS     |
| D3_36m             |           1016 |                              0 | PASS     |

## Restrictions retained

1. Report untouched OASIS-3 validation first; save and hash outcome-hidden
   predictions before any outcome merge.
2. Treat the OASIS-3 `CDRTOT` global-CDR mapping as provisional pending provider
   confirmation.
3. Keep 24- and 60-month results as sensitivities and keep endpoint
   reconstruction/recalibration secondary to untouched validation.
4. B2 and D2 remain conditional, not-saved analyses; they are not required
   members of the repaired package.
5. Do not begin Phase 5 automatically.


# Phase 4 Restart Handoff

Generated: 2026-07-29T02:37:21.308871+00:00

## Decision

**RESUME PHASE 4 WITH RESTRICTIONS**

Use this lock and no other lock:

`/media/faizaan/4TB/1_DATA_PROJECTS/2026/July clinical data adni and oasis and aibl/models/phase3/phase4_lock_v2`

The original `models/phase3/phase4_lock` remains unchanged. The repaired lock
contains byte-identical B1/D1 bundles plus ADNI-only D0, B3, and D3 bundles at
12 and 36 months. All package checksums, held-out inference fixtures, strict
D3 timing checks, and participant-grouping checks passed independently.

## Required restart sequence

1. Verify `external_validation_lock_manifest.json` and
   `checksum_manifest.csv` in the v2 lock.
2. Load only the v2 frozen index and packaged prediction runtime.
3. Create outcome-hidden OASIS-3 predictors using the frozen mapping and
   preprocessing; save and hash predictions before opening outcomes.
4. Run untouched B1, D0, D1, B3, and D3 validation at 12 and 36 months.
5. Treat B1/D1 24- and 60-month analyses as sensitivity horizons.
6. Only after untouched results are frozen may endpoint sensitivity or
   recalibration be performed.

Phase 4 external validation was not resumed by this repair or verifier.


## Final decision

In [17]:
decision = run_metadata["final_decision"]
lock_path = ROOT / run_metadata["phase4_must_use_lock"]
display(
    Markdown(
        f"# {decision}\n\n"
        f"Phase 4 must use: `{lock_path}`\n\n"
        "**Phase 4 external validation has not been resumed.**"
    )
)

# RESUME PHASE 4 WITH RESTRICTIONS

Phase 4 must use: `/media/faizaan/4TB/1_DATA_PROJECTS/2026/July clinical data adni and oasis and aibl/models/phase3/phase4_lock_v2`

**Phase 4 external validation has not been resumed.**